<a href="https://colab.research.google.com/github/polaya813/polaya813/blob/main/ETL_facts_P_Olaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conjunto de datos simulados para el dim_products

In [1]:
import pandas as pd
import numpy as np

# Generar datos simulados para productos
np.random.seed(1)
num_products = 100
data_productos = {
    'producto_id_origen': np.arange(1, num_products + 1),
    'nombre_producto': [f'Producto {i}' for i in range(1, num_products + 1)],
    'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar'], size=num_products),
    'precio': np.random.uniform(low=5, high=500, size=num_products)
}

# Crear un DataFrame
df_productos = pd.DataFrame(data_productos)

# Añadir una clave subrogada
df_productos['producto_key'] = range(1, len(df_productos) + 1)

# Guardar en un archivo CSV
df_productos.to_csv('dim_productos.csv', index=False)
print("Archivo dim_productos.csv generado con éxito.")


Archivo dim_productos.csv generado con éxito.


Conjunto simulado ventas_raw

In [3]:
import pandas as pd
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# Number of sales records to generate
num_sales = 1000

# Generate random sales data
data_ventas = {
    'venta_id': np.arange(1, num_sales + 1),
    'fecha': pd.date_range(start='2023-01-01', periods=num_sales, freq='H').strftime('%Y-%m-%d').tolist(),
    'ID de cliente': np.random.randint(1, 101, size=num_sales),
    'ID de producto': np.random.randint(1, 101, size=num_sales),
    'cantidad vendida': np.random.randint(1, 11, size=num_sales),
    'total venta': np.random.uniform(10, 1000, size=num_sales).round(2)
}

# Create a DataFrame
df_ventas = pd.DataFrame(data_ventas)

# Save to a CSV file
df_ventas.to_csv('ventas_raw.csv', index=False)
print("Archivo ventas_raw.csv generado con éxito.")


Archivo ventas_raw.csv generado con éxito.


/tmp/ipykernel_4535/3301824526.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'fecha': pd.date_range(start='2023-01-01', periods=num_sales, freq='H').strftime('%Y-%m-%d').tolist(),


Asegúrate de tener los archivos dim_clientes.csv y dim_productos.csv disponibles en tu entorno. Estos archivos deben contener las claves subrogadas necesarias para el proceso de "lookup".

In [4]:
import pandas as pd

# --- EXTRACT ---
def extract_ventas_raw(filepath='ventas_raw.csv'):
    """Extract raw sales data from a CSV file."""
    try:
        df_ventas_raw = pd.read_csv(filepath, parse_dates=['fecha'])
        print("Datos de ventas extraídos con éxito.")
        return df_ventas_raw
    except Exception as e:
        print(f"Error al extraer datos de ventas: {e}")
        return None

# --- TRANSFORM ---
def transform_f_ventas(df_ventas_raw, dim_clientes_df, dim_productos_df):
    """Transform raw sales data to match the fact table structure."""
    # Merge para obtener cliente_key
    df_transformed = pd.merge(df_ventas_raw,
                              dim_clientes_df[['cliente_id_origen', 'cliente_key']],
                              left_on='ID de cliente',
                              right_on='cliente_id_origen', how='left')

    # Merge para obtener producto_key
    df_transformed = pd.merge(df_transformed,
                              dim_productos_df[['producto_id_origen', 'producto_key']],
                              left_on='ID de producto',
                              right_on='producto_id_origen', how='left')

    # Generar fecha_key (ej. YYYYMMDD a partir de df_ventas_raw['fecha'])
    df_transformed['fecha_key'] = df_transformed['fecha'].dt.strftime('%Y%m%d').astype(int)

    # Renombrar columnas
    df_transformed.rename(columns={'cantidad vendida': 'cantidad_vendida', 'total venta': 'monto_total_venta'}, inplace=True)

    # Seleccionar y reordenar columnas para fact_ventas
    fact_cols = ['fecha_key', 'cliente_key', 'producto_key', 'cantidad_vendida', 'monto_total_venta']
    df_fact_ventas = df_transformed[fact_cols].dropna(subset=['cliente_key', 'producto_key', 'fecha_key'])

    print("Datos de ventas transformados con éxito.")
    return df_fact_ventas

# --- LOAD ---
def load_fact_ventas(df, output_filepath='fact_ventas.csv'):
    """Load transformed sales data into a CSV file."""
    try:
        df.to_csv(output_filepath, index=False)
        print(f"Datos cargados exitosamente en el archivo {output_filepath}.")
    except Exception as e:
        print(f"Error al cargar datos en el archivo: {e}")

# --- Ejecución del Pipeline para Hechos ---
def main():
    # Extraer datos de ventas
    df_ventas_raw = extract_ventas_raw()

    # Simular la carga de dimensiones (en un entorno real, esto vendría de tu DWH)
    dim_clientes_df = pd.read_csv('dim_clientes.csv')  # Asegúrate de tener este archivo
    dim_productos_df = pd.read_csv('dim_productos.csv')  # Asegúrate de tener este archivo

    if df_ventas_raw is not None:
        # Transformar datos de ventas
        df_fact_ventas = transform_f_ventas(df_ventas_raw, dim_clientes_df, dim_productos_df)
        # Cargar datos transformados
        load_fact_ventas(df_fact_ventas)

if __name__ == '__main__':
    main()


Datos de ventas extraídos con éxito.
Datos de ventas transformados con éxito.
Datos cargados exitosamente en el archivo fact_ventas.csv.
